In [121]:
import torch
import re
from collections import defaultdict

In [122]:
text = open("train.txt", "r", encoding="utf-8").read()

unique_chars = sorted(list(set(text)))
vocab_size = len(unique_chars)


print(''.join(unique_chars))
print(f"\nvocab_Size:{vocab_size}")

# unique_chars.extend(['4','6','7','9','0'])
# Adding remaining numbers 

def word_tokenize(text):
    """Each word is a token, but also each special character, spaces arent ocnsidered."""
    # Split on spaces but keep punctuation as separate tokens
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens
word_tokens = word_tokenize(text)


print(f"\nFirst 50 WORD tokens:\n {word_tokens[:50]}")




 !"&'(),-./12358:;?ABCDEFGHIJKLMNOPQRSTUVWYabcdefghijklmnopqrstuvwxyzé

vocab_Size:71

First 50 WORD tokens:
 ['A', 'SCANDAL', 'IN', 'BOHEMIA', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'Chapter', '1', 'Chapter', '2', 'Chapter', '3', 'CHAPTER', 'I', 'To', 'Sherlock', 'Holmes', 'she', 'is', 'always', 'the', 'woman', '.', 'I', 'have', 'seldom', 'heard', 'him', 'mention', 'her', 'under', 'any', 'other', 'name', '.', 'In', 'his', 'eyes', 'she', 'eclipses', 'and', 'predominates', 'the', 'whole', 'of', 'her']


In [123]:
def split_words_with_sep(text):
    for m in re.finditer(r'(\S+)(\s*)', text):
        word, sep = m.groups()
        if not word:
            continue
        marker = '<para>' if sep.count('\n') >= 2 else '<end>'
        yield word, marker

def get_vocab(text):
    vocab = defaultdict(int)
    for word, marker in split_words_with_sep(text):
        vocab[' '.join(list(word)) + ' ' + marker] += 1
    return vocab

def get_stats(vocab):
    """Count frequency of every adjacent pair and stores in dict pairs"""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_mostfrequent(best_pair, vocab):
    """Merge the most frequent pair across all words, variable- pair is the most freq adjacent pair"""
    new_vocab = {}
    to_replace = ' '.join(best_pair)
    replacement = ''.join(best_pair)
    for word in vocab:
        new_word = word.replace(to_replace, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab


def train_bpe(text, n_merges):
    vocab = get_vocab(text)
    merges = []

    for i in range(n_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        # merge the most frequent pair
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_mostfrequent(best_pair, vocab)
        merges.append(best_pair)

    return vocab, merges

print("\n\n")
bpe_vocab, merges = train_bpe(text, n_merges=1000)

# Build final token list from BPE vocab
bpe_tokens = set()
for word in bpe_vocab:
    for token in word.split():
        bpe_tokens.add(token)

bpe_tokens = sorted(list(bpe_tokens))
print(f"BPE vocab size: {len(bpe_tokens)}")
print(f"First 20 tokens: {bpe_tokens[:20]}")

stoi = {token: i for i, token in enumerate(bpe_tokens)}
itos = {i: token for i, token in enumerate(bpe_tokens)}

bpe_vocab_size=len(bpe_tokens)
#string ot integer and integer to string

def encode(text):
    token_ids = []
    for word, marker in split_words_with_sep(text):
        symbols = list(word) + [marker]
        for pair in merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                    symbols = symbols[:i] + [''.join(pair)] + symbols[i+2:]
                else:
                    i += 1
        for token in symbols:
            if token in stoi:
                token_ids.append(stoi[token])
    return token_ids

encoded = encode(text)

def decode(token_ids):
    tokens = [itos[i] for i in token_ids]
    text = ''.join(tokens)
    text = text.replace('<para>', '\n\n').replace('<end>', ' ')
    return text.strip()

n = int(0.9 * len(encoded))
train = encoded[:n]
test = encoded[n:]


train = torch.tensor(train, dtype=torch.long)
test = torch.tensor(test, dtype=torch.long)

print(f"Train tokens: {len(train)}, Test tokens: {len(test)}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")






BPE vocab size: 2539
First 20 tokens: ['!', '!"<end>', '!"<para>', '!<end>', '"', '"\'', '"<end>', '"<para>', '"A', '"A<end>', '"And<end>', '"At<end>', '"But<end>', '"C', '"F', '"Fi', '"From<end>', '"H', '"He<end>', '"How<end>']
Train tokens: 17607, Test tokens: 1957
Device: CPU


In [124]:
torch.manual_seed(10)
block_size = 8   # context length 
batch_size = 4   #no of sampleing chunks in one forward pass

def get_batch(data):
    #random start positions
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # x is the input context, y is the target (shifted by 1)
    x = torch.stack([data[i : i+block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    
    return x, y

xb, yb = get_batch(train)
print(xb.shape)
print(yb.shape,"\n\n")

print("INPUTS",xb)
print("OUTPUTS:",yb)

    

torch.Size([4, 8])
torch.Size([4, 8]) 


INPUTS tensor([[2229, 1206, 1939, 2226, 2477,  663,  565,  549],
        [1641, 2264,  907,  383, 1385, 1134,  921, 1243],
        [2229, 1699, 1835, 1932,  965,   80,  225, 1824],
        [ 243, 2226, 1676,   72,  903, 2419,  446, 1932]])
OUTPUTS: tensor([[1206, 1939, 2226, 2477,  663,  565,  549, 2398],
        [2264,  907,  383, 1385, 1134,  921, 1243, 2303],
        [1699, 1835, 1932,  965,   80,  225, 1824,  903],
        [2226, 1676,   72,  903, 2419,  446, 1932,  713]])


In [125]:
torch.manual_seed(10)
import torch.nn as nn  

import torch.nn.functional as F
n_embed=32

In [126]:
# Self Attention Head:
# Every token spits out two vectors, one is the key vector and the other is the query vector. The key vector is used to represent the token itself, 
# the query vector is used to represent the token's relationship with prev toks.
# Query . Key(eachtoken) gives a score that represents how much attention should be paid to each token when predicting the next token. 
#  score is softmaxxed to get PDF over all tokens.
# This is  used to compute a weighted sum of the value vectors of all tokens, which gives the final output for that token.

# weight matrix becomes that once our head is done.

In [127]:
#Attention HEAD.
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        weight = q @ k.transpose(-2, -1) * (C**-0.5)            # (B, T, T)
        weight = weight.masked_fill(self.tril[:T,:T]==0, float('-inf'))
        weight = F.softmax(weight, dim=-1)                        # (B, T, T)

        v = self.value(x)  # (B, T, head_size)
        return weight @ v  # (B, T, head_size)

In [128]:
class BigramLM(nn.Module):
    
    def __init__(self, vocab_size, n_embed):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed) # each pos from 0 to block_size gets its own embedding vector
        self.head = Head(n_embed)        # single self-attention head, head_size = n_embed = 32
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, index, targets=None):
        # index is B x T array of indices in the current context
        tok_emb = self.token_embedding_table(index)                                                # (B, T, n_embed)
        pos_emb = self.position_embedding_table(torch.arange(index.size(1), device=index.device)) # (T, n_embed)
        x = tok_emb + pos_emb                                                                      # (B, T, n_embed)
        x = self.head(x)                                                                           # (B, T, n_embed) — tokens now attend to each other
        logits = self.lm_head(x)                                                                   # (B, T, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is B x T array of indices in the current context
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]                        # crop to block_size so position embeddings don't go out of range
            logits, _ = self.forward(index_cond)
            logits = logits[:, -1, :]                                  # shape (B, C) — only last token's prediction
            probs = F.softmax(logits, dim=-1)                          # shape (B, C)
            next_token = torch.multinomial(probs, num_samples=1)       # shape (B, 1)
            index = torch.cat([index, next_token], dim=1)              # append, repeat
        return index
    
model = BigramLM(bpe_vocab_size, n_embed)
logits,loss=model(xb,yb)
print(logits.shape)
print(loss)   #lideally loss should be -ln(1/bpe_vocab_size)
#output is a PREDICTION score for EACH element in the torch.stack 2d array, for each unique token, it gives a UN-NORMALISED probability that thats the next token


torch.Size([32, 2539])
tensor(7.9038, grad_fn=<NllLossBackward0>)


In [134]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32
best_loss =2.6513

for i in range(1000):
    xb, yb = get_batch(train)
    logits, loss = model(xb, yb)    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), 'bigram_weights.pth')

    if i%1000==0:
        print(f"loss {loss.item():.4f}, min is {best_loss:.4f}")

print(f"Training done. Best loss: {best_loss:.4f}")

loss 2.8857, min is 2.6513
Training done. Best loss: 2.6513


In [130]:
# To capture more info about previous tokens, we can use multiple ways.
#  We can aggregate the mean of all previous tokens instead of just using the previous token for predicting the next token,

# xagg=torch.zeros((B,T,C))
# for b in range(B):
#     for t in range(T):
#         xprev=x[b,:t+1,:]
#         xagg[b,t]=torch.mean(xprev,dim=0)

# xagg[b,t] stores all previous info till t (excluding t). This is a simple way to give the model more context, but it can be improved.

# print(x[0],"\n\n",xagg[0])



#This can be done using matrix multiplicaiton or softmax in a much more efficient matter instead of adding.

In [131]:
# torch.manual_seed(10)
# B,T,C=4,8,32
# x=torch.randn(B,T,C)
# x.shape

# tril=torch.tril(torch.ones(T,T))
# weight=torch.zeros((T,T))
# weight=weight.masked_fill(tril==0,float('-inf'))  # future values are nerfed because we only concentrate till present token.
# weight=F.softmax(weight,dim=-1)
# out=weight@x

# out.shape 

In [136]:
model.load_state_dict(torch.load('bigram_weights.pth'))
model.eval()

context = torch.tensor([encode("Sherlock Holmes")], dtype=torch.long)
generated = model.generate(context, max_new_tokens=200)
print(decode(generated[0].tolist()))

Sherlock es in quat when ey. The ly."

"How official remed. I heard frome and heavy mont six wholuttriges and obreet.

His manth her than instrufentic age action pull thoin it ild had ton."

A akat the coay, herver Sely. clushed and laid outine and hurfluence for the ter of looking pap.

eone of binly, with a lout serts."

"P," crimeenll. I was already been lie and ter opas! Chapo not love you, hout arm. When too way by Monare ain," I ped Nestit in silk his inusus, ioutorle s
